In [13]:
"""
Digital Earth Australia Coastline widget, which can be used to 
interactively extract shoreline data using transects.
"""

# Import required packages
import fiona
import os
import sys
import datacube
import warnings
import matplotlib.pyplot as plt
from datacube.utils.geometry import CRS
from ipyleaflet import (
    WMSLayer,
    basemaps,
    basemap_to_tiles,
    Map,
    DrawControl,
    WidgetControl,
    LayerGroup,
    LayersControl,
    GeoData,
)
from traitlets import Unicode
from ipywidgets import (
    GridspecLayout,
    Button,
    Layout,
    HBox,
    VBox,
    HTML,
    Output,
)
import json
import geopandas as gpd
from io import BytesIO
import ipywidgets as widgets

sys.path.insert(1, "../Tools/")
import dea_tools.app.widgetconstructors as deawidgets
from dea_tools.wetlands import generate_low_quality_data_periods
from dea_tools.wit import WIT_drill


def make_box_layout():
    return Layout(
        #          border='solid 1px black',
        margin='0px 10px 10px 0px',
        padding='5px 5px 5px 5px',
        width='100%',
        height='100%',
    )


def create_expanded_button(description, button_style):
    return Button(
        description=description,
        button_style=button_style,
        layout=Layout(width="auto", height="auto"),
    )


class transect_app(HBox):

    def __init__(self):
        super().__init__()

        ######################
        # INITIAL ATTRIBUTES #
        ######################

        self.startdate = "2024-01-01"
        self.enddate = "2024-03-01"
        self.out_csv = "example_WIT.csv"
        self.out_plot = "example_WIT.png"
        self.product_list = [
            ("ESRI World Imagery", "none"),
            ("Open Street Map", "open_street_map"),
        ]
        self.product = self.product_list[0][1]
        self.target = None
        self.action = None
        self.gdf_drawn = None
        self.gdf_uploaded = None
        self.mingooddata = 0.0
        self.resamplingfreq = "1M"
        self.wetlandname = ""

        ##################
        # HEADER FOR APP #
        ##################

        # Create the Header widget
        header_title_text = "<h3>Digital Earth Australia Coastlines shoreline transect extraction</h3>"
        instruction_text = "Select parameters and draw a transect on the map to extract shoreline data. <b>In distance mode</b>, draw a transect line starting from land that crosses multiple shorelines. <br><b>In width mode</b>, draw a transect line that intersects shorelines at least twice. Alternatively, <b>upload an vector file</b> to extract shoreline data for multiple existing transects."
        self.header = deawidgets.create_html(
            f"{header_title_text}<p>{instruction_text}</p>")
        self.header.layout = make_box_layout()

        #####################################
        # HANDLER FUNCTION FOR DRAW CONTROL #
        #####################################

        # Define the action to take once something is drawn on the map
        def update_geojson(target, action, geo_json):

            # Remove previously uploaded data if present
            self.gdf_uploaded = None
            fileupload_transects._counter = 0

            # Get data from action
            self.action = action

            # Convert data to geopandas
            json_data = json.dumps(geo_json)
            binary_data = json_data.encode()
            io = BytesIO(binary_data)
            io.seek(0)
            gdf = gpd.read_file(io)
            gdf.crs = "EPSG:4326"

            # Convert to Albers and compute area
            gdf_drawn_albers = gdf.copy().to_crs("EPSG:3577")
            m2_per_km2 = 10**6
            area = gdf_drawn_albers.envelope.area.values[0] / m2_per_km2
            polyarea_label = 'Total area of DEA Coastlines data to extract'
            polyarea_text = f"<b>{polyarea_label}</b>: {area:.2f} km<sup>2</sup>"

            # Test area size
            if area <= 50000:
                confirmation_text = '<span style="color: #33cc33"> <b>(Area to extract falls within recommended limit; click "Extract shoreline data" to continue)</b></span>'
                self.header.value = header_title_text + polyarea_text + confirmation_text
                self.gdf_drawn = gdf
            else:
                warning_text = '<span style="color: #ff5050"> <b>(Area to extract is too large, please select a smaller transect)</b></span>'
                self.header.value = header_title_text + polyarea_text + warning_text
                self.gdf_drawn = None

        ###########################
        # WIDGETS FOR APP OUTPUTS #
        ###########################

        self.status_info = Output(layout=make_box_layout())
        self.output_plot = Output(layout=make_box_layout())

        #########################################
        # MAP WIDGET, DRAWING TOOLS, WMS LAYERS #
        #########################################

        # Create drawing tools
        desired_drawtools = ["rectangle", "polygon"]
        draw_control = deawidgets.create_drawcontrol(desired_drawtools)


        # Begin by displaying an empty layer group, and update the group with desired WMS on interaction.
        self.map_layers = LayerGroup(layers=())
        self.map_layers.name = 'Map Overlays'

        # Create map widget
        self.m = deawidgets.create_map(map_center=(-28, 135),
                                       zoom_level=4,
                                       basemap=basemaps.Esri.WorldImagery)
        self.m.layout = make_box_layout()

        # Add tools to map widget
        self.m.add_control(draw_control)
        self.m.add_layer(self.map_layers)

        # Store current basemap for future use
        self.basemap = self.m.basemap

        ############################
        # WIDGETS FOR APP CONTROLS #
        ############################

        # Create parameter widgets
        startdate_picker = deawidgets.create_datepicker()
        
        enddate_picker = deawidgets.create_datepicker()

        output_csv = deawidgets.create_inputtext(self.out_csv, self.out_csv)
        
        output_plot = deawidgets.create_inputtext(self.out_plot, self.out_plot)

        deaoverlay_dropdown = deawidgets.create_dropdown(
            self.product_list, self.product_list[0][1])

        min_good_data = deawidgets.create_boundedfloattext(self.mingooddata, 0.0, 1.0, 0.05)
        
        resampling_freq = deawidgets.create_inputtext(self.resamplingfreq, self.resamplingfreq)

        run_button = create_expanded_button("Extract shoreline data", "info")
        fileupload_transects = widgets.FileUpload(accept='', multiple=True)

        ####################################
        # UPDATE FUNCTIONS FOR EACH WIDGET #
        ####################################

        # Run update functions whenever various widgets are changed.
        startdate_picker.observe(self.update_startdate, "value")
        enddate_picker.observe(self.update_enddate, "value")

        output_csv.observe(self.update_outputcsv, "value")
        output_plot.observe(self.update_outputplot, "value")
        min_good_data.observe(self.update_mingooddata, "value")
        resampling_freq.observe(self.update_resamplingfreq, "value")
        deaoverlay_dropdown.observe(self.update_deaoverlay, "value")
        run_button.on_click(self.run_app)
        draw_control.on_draw(update_geojson)
        fileupload_transects.observe(self.update_fileupload_transects, "value")

        ##################################
        # COLLECTION OF ALL APP CONTROLS #
        ##################################

        parameter_selection = VBox([
            HTML("<bstate date:</b>"), startdate_picker,
            HTML("<b>" + ("End Date:") + "</b>"),
                enddate_picker,
            HTML("<b>" + ("Minimum Good Data:") + "</b>"),
                min_good_data,
            HTML("<b>" + ("Resampling Frequency:") + "</b>"),
                resampling_freq,
            HTML("<b>" + ("Output CSV:") + "</b>"),
                output_csv,
            HTML("<b>" + ("Output Plot:") + "</b>"),
                output_plot,
            HTML(
                "</br><i><b>Advanced</b></br>Upload a GeoJSON or ESRI "
                "Shapefile (<5 mb) containing one or more transect lines.</i>"),
            fileupload_transects
        ])
        map_selection = VBox([
            HTML("</br><b>Map overlay:</b>"),
            deaoverlay_dropdown,
        ])
        parameter_selection.layout = make_box_layout()
        map_selection.layout = make_box_layout()

        ###############################
        # SPECIFICATION OF APP LAYOUT #
        ###############################

        #       0   1    2   3   4   5   6   7    8   9
        #     ---------------------------------------------
        # 0   | Header                         | Map sel. |
        #     ---------------------------------------------
        # 1   | Params |                                  |
        # 2   |        |                                  |
        # 3   |        |                                  |
        # 4   |        |               Map                |
        # 5   |        |                                  |
        #     ----------                                  |
        # 6   |  Run   |                                  |
        #     ---------------------------------------------
        # 7   |               Status info                 |
        #     ---------------------------------------------
        # 8   |                                           |
        # 9   |               Output/figure               |
        # 10  |                                           |
        # 11  | ------------------------------------------|

        # Create the layout #[rowspan, colspan]
        grid = GridspecLayout(12, 10, height="1350px", width="auto")

        # Header and controls
        grid[0, :8] = self.header
        grid[0, 8:] = map_selection
        grid[1:6, 0:2] = parameter_selection
        grid[6, 0:2] = run_button

        # Status info, map and plot
        grid[1:7, 2:] = self.m  # map
        grid[7:8, :] = self.status_info
        grid[8:, :] = self.output_plot

        # Display using HBox children attribute
        self.children = [grid]

    ######################################
    # DEFINITION OF ALL UPDATE FUNCTIONS #
    ######################################

    # Set the output csv
    def update_fileupload_transects(self, change):

        # Clear any drawn data if present
        self.gdf_drawn = None
        
        # Temporary compatibility fix for ipywidget > 8.0
        # TODO: Update code to use new fileupload API documented here:
        # https://ipywidgets.readthedocs.io/en/latest/user_migration_guides.html#fileupload
        uploaded_data = {f["name"]: {"content": f.content.tobytes()} for f in change.new}            

        # Save to file
        for uploaded_filename in uploaded_data.keys():
            with open(uploaded_filename, "wb") as output_file:
                content = uploaded_data[uploaded_filename]["content"]
                output_file.write(content)

        with self.status_info:

            try:            

                print('Loading vector data...', end='\r')
                valid_files = [
                    file for file in uploaded_data.keys()
                    if file.lower().endswith(('.shp', '.geojson'))
                ]
                valid_file = valid_files[0]
                transect_gdf = (gpd.read_file(valid_file).to_crs(
                    "EPSG:4326").explode(index_parts=True).reset_index(drop=True))

                # Use ID column if it exists
                if 'id' in transect_gdf:
                    transect_gdf = transect_gdf.set_index('id')
                    print(f"Uploaded '{valid_file}'; automatically labelling "
                          "transects using column 'id'.")
                else:
                    print(
                        f"Uploaded '{valid_file}'; no 'id' column detected, "
                        f"labelling transects from 0 to {len(transect_gdf.index) - 1}."
                    )

                # Create a geodata
                geodata = GeoData(geo_dataframe=transect_gdf,
                                  style={
                                      'color': 'black',
                                      'weight': 3
                                  })

                # Add to map
                xmin, ymin, xmax, ymax = transect_gdf.total_bounds
                self.m.fit_bounds([[ymin, xmin], [ymax, xmax]])
                self.m.add_layer(geodata)

                # If completed, add to attribute
                self.gdf_uploaded = transect_gdf

            except IndexError:
                print(
                    "Cannot read uploaded files. Please ensure that data is "
                    "in either GeoJSON or ESRI Shapefile format.",
                    end='\r')
                self.gdf_uploaded = None

            except fiona.errors.DriverError:
                print(
                    "Shapefile is invalid. Please ensure that all shapefile "
                    "components (e.g. .shp, .shx, .dbf, .prj) are uploaded.",
                    end='\r')
                self.gdf_uploaded = None

    # set the start date to the new edited date
    def update_startdate(self, change):
        self.startdate = change.new

    # set the end date to the new edited date
    def update_enddate(self, change):
        self.enddate = change.new

    # set the min good data
    def update_mingooddata(self, change):
        self.mingooddata = change.new

    # set the resampling frequency
    def update_resamplingfreq(self, change):
        self.resamplingfreq = change.new

    # set the output csv
    def update_outputcsv(self, change):
        self.out_csv = change.new

    # set the output plot
    def update_outputplot(self, change):
        self.out_plot = change.new

    # Update product
    def update_deaoverlay(self, change):

        self.product = change.new

        if self.product == "none":
            self.map_layers.clear_layers()
            self.map_layers.add_layer(deacoastlines)

        elif self.product == "open_street_map":
            self.map_layers.clear_layers()
            layer = basemap_to_tiles(basemaps.OpenStreetMap.Mapnik)
            self.map_layers.add_layer(layer)
            self.map_layers.add_layer(deacoastlines)

    def run_app(self, change):

        # Clear progress bar and output areas before running
        self.status_info.clear_output()
        self.output_plot.clear_output()

        # Run DEA Coastlines analysis
        with self.status_info:
            warnings.filterwarnings("ignore")

            # Load transects from either map or uploaded files
            if self.gdf_uploaded is not None:
                transect_gdf = self.gdf_uploaded
                run_text = 'uploaded file'
            elif self.gdf_drawn is not None:
                transect_gdf = self.gdf_drawn
                transect_gdf.index = [self.output_name]
                run_text = 'selected transect'
            else:
                print(f'No transect drawn or uploaded. Please select a transect on the map, or upload a GeoJSON or ESRI Shapefile.',
                      end='\r')
                transect_gdf = None

            
 # Load polygons from either map or uploaded files
            if self.gdf_uploaded is not None:
                wetlands_gdf = self.gdf_uploaded
                run_text = 'uploaded file'
            elif self.gdf_drawn is not None:
                wetlands_gdf = self.gdf_drawn
                #wetlands_gdf.index = [self.output_name]
                run_text = 'selected polygon'
            else:
                print(f'No transect drawn or uploaded. Please select a transect on the map, or upload a GeoJSON or ESRI Shapefile.',
                      end='\r')
                wetlands_gdf = None

            dask_chunks = dict(x=1000, y=1000, time=1)

                    # check resampling freq
            if self.resamplingfreq == "None":
                rsf = None
            else:
                rsf = self.resamplingfreq
                
            try:
                df = WIT_drill(
                    gdf=wetlands_gdf,
                    time=(self.startdate, self.enddate),
                    min_gooddata=self.mingooddata,
                    resample_frequency=rsf,
                    #TCW_threshold=TCW_threshold,
                    export_csv=self.out_csv,
                    dask_chunks=dask_chunks,
                    verbose=False,
                    verbose_progress=True,
                )
                print(("WIT complete"))
            except AttributeError:
                print(("No polygon selected"))

        # close down the dask client
        client.shutdown()

        # save the csv
        if self.out_csv:
            df.to_csv(self.out_csv, index_label="Datetime")

        # ---Plotting------------------------------

        with self.wit_plot:

            fontsize = 17
            plt.rcParams.update({"font.size": fontsize})
            # set up color palette
            pal = [
                sns.xkcd_rgb["cobalt blue"],
                sns.xkcd_rgb["neon blue"],
                sns.xkcd_rgb["grass"],
                sns.xkcd_rgb["beige"],
                sns.xkcd_rgb["brown"],
            ]

            # make a stacked area plot
            plt.close("all")

            fig, ax = plt.subplots(constrained_layout=True, figsize=(20, 6))

            ax.stackplot(
                df["date"],
                df["water"] * 100,
                df["wet"] * 100,
                df["norm_pv"] * 100,
                df["norm_npv"] * 100,
                df["norm_bs"] * 100,
                colors=pal,
                alpha=0.7,
            )

            # manually change the legend display order
            legend = ax.legend(
                ["open water", "wet", "green veg", "dry veg", "bare soil"][::-1],
                loc="lower left",
            )
            handles = legend.legend_handles
        
            for i, handle in enumerate(handles):
                handle.set_facecolor(pal[::-1][i])
                handle.set_alpha(0.7)
        
            # setup the display ranges
            ax.set_ylim(0, 100)
            ax.set_xlim(df["date"].min(), df["date"].max())
        
            # add a new column: 'off_value' based on low quality data setting
            df = generate_low_quality_data_periods(df)
        
            ax.fill_between(
                df["date"],
                0,
                100,
                where=df["off_value"] == 100,
                color="white",
                alpha=0.5,
                hatch="//",
            )

                
            x_label_text = "The Fractional Cover algorithm developed by the Joint Remote Sensing Research Program and\n the Water Observations from Space algorithm developed by Geoscience Australia are used in the production of this data"
        
            ax.set_xlabel(x_label_text, style="italic")
         
            # add a legend and a tight plot box
            #ax.legend(loc="lower left", framealpha=0.6)
            wetlandname = self.wetlandname
            #ax.set_title(wetlandname, fontsize='large', pad=20)
            #ax.text(0.5, 1.02, "Percentage Fractional Cover, Wetness, and Water", 
        #transform=ax.transAxes, ha='center', fontsize='medium')  # Subtitle
            ax.set_title(("Percentage Fractional Cover, Wetness, and Water"))
            # plt.tight_layout()
            plt.show()

            if self.out_plot:
                # save the figure
                fig.savefig(f"{self.out_plot}")

  

In [14]:
transect_app()


transect_app(children=(GridspecLayout(children=(HTML(value='<h3>Digital Earth Australia Coastlines shoreline t…